## Imports e configuração

In [4]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import pyarrow.dataset as ds
import numpy as np
import os
import gc
import pyarrow


os.makedirs("../outputs/processed", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)


## Leitura dos dados processados

In [5]:
deng = glob.glob("../data/raw/dengue_sinan/*.parquet")
chik = glob.glob("../data/raw/chikungunya_sinan/*.parquet")
pop = "../data/raw/municipios_ibge/POP2025_20260113.xls"


In [12]:
df = pd.read_excel(pop, skiprows=1, sheet_name="Municípios")
df.head()

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA,Unnamed: 5
0,RO,11,15.0,Alta Floresta D'Oeste,22787.0,NaN
1,RO,11,23.0,Ariquemes,109170.0,NaN
2,RO,11,31.0,Cabixi,5664.0,NaN
3,RO,11,49.0,Cacoal,98280.0,NaN
4,RO,11,56.0,Cerejeiras,16966.0,NaN


## Contagem total de casos

In [3]:
contagem= 0
contagem2 = 0
for i in deng:
    dataset = ds.dataset(i, format="parquet")
    contagem += dataset.count_rows()


for i in chik:
    dataset = ds.dataset(i, format="parquet")
    contagem2 += dataset.count_rows()

print(f"Dengue: {contagem} Chik: {contagem2}")

Dengue: 33225462 Chik: 2376910


## Contagem de nulos

In [ ]:
def analisar_nulos(base, outputname):
    resultados = []

    for indice in base:
        df = pd.read_parquet(indice, engine="pyarrow")
        df = df.replace(r"^\s*$", np.nan, regex=True)
        total_linhas = len(df)

        linha = {"parquet": os.path.basename(indice)}

        for col in df.columns:
            nulos = df[col].isnull().sum()
            porcentagem = 0 if total_linhas == 0 else (nulos / total_linhas) * 100
            linha[f"{col}_%"] = round(porcentagem, 2)

        resultados.append(linha)
        del df
        gc.collect()

    df_final = pd.DataFrame(resultados)
    caminho = f"../outputs/processed/{outputname}.xlsx"
    df_final.to_excel(caminho, index=False)
    print(f"Salvo: {caminho}")


analisar_nulos(deng, "analise_nulos_dengue")
analisar_nulos(chik, "analise_nulos_chikungunya")


## Diagnóstico de ID de município

In [ ]:
def diagnostico_id_municip(base, nome_base):
    contador_total = pd.Series(dtype=int)

    for arquivo in base:
        df = pd.read_parquet(arquivo, columns=["ID_MUNICIP"])
        df["ID_MUNICIP"] = df["ID_MUNICIP"].astype(str).str.strip()
        contagem = df["ID_MUNICIP"].str.len().value_counts()
        contador_total = contador_total.add(contagem, fill_value=0)
        del df

    contador_total = contador_total.sort_index()
    print(f"\nDiagnóstico ID_MUNICIP — {nome_base}")
    print(contador_total)
    return contador_total


diag_deng = diagnostico_id_municip(deng, "Dengue")
diag_chik = diagnostico_id_municip(chik, "Chikungunya")

# salva diagnóstico combinado
df_diag = pd.DataFrame({
    "comprimento_codigo": diag_deng.index.union(diag_chik.index),
})
df_diag["dengue"]      = df_diag["comprimento_codigo"].map(diag_deng).fillna(0).astype(int)
df_diag["chikungunya"] = df_diag["comprimento_codigo"].map(diag_chik).fillna(0).astype(int)

caminho_diag = "../outputs/processed/diagnostico_id_municip.csv"
df_diag.to_csv(caminho_diag, index=False)
print(f"\nSalvo: {caminho_diag}")


## Mapeamento dos municípios

In [3]:
import pandas as pd


def processar_municipios(base, nome):

    caminhotab = "../data/raw/municipios_ibge/Tabela_municipios.xls"

    base = sorted(base)

    df_mun = pd.read_excel(caminhotab, skiprows=6)

    df_mun.columns = df_mun.columns.str.strip()

    COL_COD = "Código Município Completo"
    COL_NOME = "Nome_Município"

    df_mun["ID_MUNICIP"] = df_mun[COL_COD].astype(str).str.zfill(7)
    df_mun["COD6"] = df_mun["ID_MUNICIP"].str[:6]

    mapa_mun = dict(zip(df_mun["COD6"], df_mun[COL_NOME]))

    regioes = {
        "1": "Norte",
        "2": "Nordeste",
        "3": "Sudeste",
        "4": "Sul",
        "5": "Centro-Oeste"
    }

    lista = []
    total_lidos = 0

    for i in base:

        print(f"Processando: {i}")

        df = pd.read_parquet(
            i,
            columns=[
                "NU_ANO",
                "ID_MUNICIP",
                "DT_SIN_PRI"
            ]
        )

        df["ID_MUNICIP"] = df["ID_MUNICIP"].astype(str).str.strip()

        total_lidos += len(df)

        df = df[df["ID_MUNICIP"].str.len().isin([6, 7])]

        df["COD6"] = df["ID_MUNICIP"].str[:6]

        df["Município"] = df["COD6"].map(mapa_mun)

        df["Região"] = df["COD6"].str[0].map(regioes)

        df = df.dropna(subset=["Município"])

        lista.append(df)

        del df

    df_final = pd.concat(lista, ignore_index=True)

    caminho_saida = f"../outputs/processed/{nome}_municipios.parquet"

    df_final.to_parquet(caminho_saida, index=False)

    return df_final


processar_municipios(deng, "dengue")
processar_municipios(chik, "chik")

Processando: ../data/raw/dengue_sinan/DENGBR00.parquet
Processando: ../data/raw/dengue_sinan/DENGBR01.parquet
Processando: ../data/raw/dengue_sinan/DENGBR02.parquet
Processando: ../data/raw/dengue_sinan/DENGBR03.parquet
Processando: ../data/raw/dengue_sinan/DENGBR04.parquet
Processando: ../data/raw/dengue_sinan/DENGBR05.parquet
Processando: ../data/raw/dengue_sinan/DENGBR06.parquet
Processando: ../data/raw/dengue_sinan/DENGBR07.parquet
Processando: ../data/raw/dengue_sinan/DENGBR08.parquet
Processando: ../data/raw/dengue_sinan/DENGBR09.parquet
Processando: ../data/raw/dengue_sinan/DENGBR10.parquet
Processando: ../data/raw/dengue_sinan/DENGBR11.parquet
Processando: ../data/raw/dengue_sinan/DENGBR12.parquet
Processando: ../data/raw/dengue_sinan/DENGBR13.parquet
Processando: ../data/raw/dengue_sinan/DENGBR14.parquet
Processando: ../data/raw/dengue_sinan/DENGBR15.parquet
Processando: ../data/raw/dengue_sinan/DENGBR16.parquet
Processando: ../data/raw/dengue_sinan/DENGBR17.parquet
Processand

,NU_ANO,ID_MUNICIP,DT_SIN_PRI,COD6,Município,Região
0,2015,280460,20150101,280460,Nossa Senhora das Dores,Nordeste
1,2015,240310,20150101,240310,Currais Novos,Nordeste
2,2015,260890,20150101,260890,Limoeiro,Nordeste
3,2015,292630,20150101,292630,Riachão do Jacuípe,Nordeste
4,2015,292630,20141227,292630,Riachão do Jacuípe,Nordeste
...,...,...,...,...,...,...
2376905,2026,170240,20260121,170240,Arraias,Norte
2376906,2026,170030,20260114,170030,Aguiarnópolis,Norte
2376907,2026,170210,20260127,170210,Araguaína,Norte
2376908,2026,170030,20260112,170030,Aguiarnópolis,Norte


,NU_ANO,ID_MUNICIP,DT_SIN_PRI,COD6,Município,Região
0,2015,280460,20150101,280460,Nossa Senhora das Dores,Nordeste
1,2015,240310,20150101,240310,Currais Novos,Nordeste
2,2015,260890,20150101,260890,Limoeiro,Nordeste
3,2015,292630,20150101,292630,Riachão do Jacuípe,Nordeste
4,2015,292630,20141227,292630,Riachão do Jacuípe,Nordeste


In [9]:
import pandas as pd

deng_casos = pd.read_parquet("../outputs/processed/dengue_municipios.parquet")
chik_casos = pd.read_parquet("../outputs/processed/chik_municipios.parquet")


def gerar_incidencia(df_municipios, arquivo_pop, nome, pop_min=50000):

    casos = (
        df_municipios
        .groupby(["Município", "NU_ANO"])
        .size()
        .reset_index(name="CASOS")
    )

    casos["Município"] = (
        casos["Município"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    pop = pd.read_excel(
        arquivo_pop,
        skiprows=1,
        sheet_name="Municípios"
    )

    pop.columns = pop.columns.str.strip()

    pop["Município"] = (
        pop["NOME DO MUNICÍPIO"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    pop["POPULACAO"] = pd.to_numeric(
        pop["POPULAÇÃO ESTIMADA"],
        errors="coerce"
    )

    incidencia = casos.merge(
        pop[["Município", "POPULACAO"]],
        on="Município",
        how="left"
    )

    incidencia = incidencia.dropna(
        subset=["POPULACAO"]
    )

    incidencia = incidencia[
        incidencia["POPULACAO"] >= pop_min
    ]

    incidencia["INCIDENCIA_100K"] = (
        incidencia["CASOS"]
        / incidencia["POPULACAO"]
    ) * 100000

    incidencia = incidencia.sort_values(
        ["NU_ANO", "INCIDENCIA_100K"],
        ascending=[True, False]
    ).reset_index(drop=True)

    caminho_saida = (
        f"../outputs/processed/"
        f"incidencia_{nome}.parquet"
    )

    incidencia.to_parquet(
        caminho_saida,
        index=False
    )

    print(f"Arquivo salvo: {caminho_saida}")

    return incidencia

df_chik = gerar_incidencia(
    chik_casos,
    pop,
    "chikungunya",
    20000
)

Arquivo salvo: ../outputs/processed/incidencia_chikungunya.parquet


In [7]:
a = pd.read_parquet("/home/caio_cabral/Music/SINAN_CIDA/outputs/processed/incidencia_dengue.parquet")
a.head()

,Município,NU_ANO,CASOS,POPULACAO,INCIDENCIA_100K
0,MIRASSOL,1999,3,65811.0,4.558508
1,SÃO JOSÉ DO RIO PRETO,1999,17,504166.0,3.371905
2,SANTOS,1999,12,429547.0,2.793641
3,VOTUPORANGA,1999,1,100568.0,0.994352
4,SANTA BÁRBARA D'OESTE,1999,1,189456.0,0.527827
